In [1]:
import torch
import pandas as pd
import os
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import nltk
from nltk.tokenize import word_tokenize

In [2]:

base_dir = os.getcwd()
os.chdir(base_dir)
nltk_data_dir = os.path.join(base_dir, "nltk_data")
if not os.path.exists(nltk_data_dir):
    os.makedirs(nltk_data_dir)
print(base_dir)

e:\Work_Space\WorkingSpace\AGNewsClassification


### Tokenizer

In [3]:
nltk.data.path.append(nltk_data_dir)

def download_nltk_data():
    for resource in ['punkt', 'punkt_tab']:
        try:
            nltk.data.find(f'tokenizers/{resource}')
            print(f"{resource} 已存在於 {nltk_data_dir}")
        except LookupError:
            print(f"正在下載 {resource} 到 {nltk_data_dir}...")
            nltk.download(resource, download_dir=nltk_data_dir)
            print(f"{resource} 下載完成")
            
download_nltk_data()

try:
    test_text = "This is a test sentence."
    tokens = word_tokenize(test_text)
    print("分詞測試成功，結果：", tokens)
except Exception as e:
    print("分詞測試失敗：", str(e))

punkt 已存在於 e:\Work_Space\WorkingSpace\AGNewsClassification\nltk_data
punkt_tab 已存在於 e:\Work_Space\WorkingSpace\AGNewsClassification\nltk_data
分詞測試成功，結果： ['This', 'is', 'a', 'test', 'sentence', '.']


In [4]:
# Data
df_train = pd.read_csv("AG_News/train.csv")
df_test  = pd.read_csv("AG_News/test.csv" )
class_map = {}

with open("AG_News/classes_name.txt", "r", encoding="utf-8") as f:
    for line in f:
        idx, label = line.strip().split(" ", 1)
        class_map[int(idx)-1] = label

df_train['Class Index'] = df_train['Class Index'] - 1
df_test['Class Index']  = df_test['Class Index'] - 1

# Combine Title & Description
df_train['text'] = df_train['Title'] + ' ' + df_train['Description']
df_test['text']  = df_test ['Title'] + ' ' + df_test ['Description']

class_map

{0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}

### Dataset

In [5]:
# Params / Tools Prepare
device = torch.device('cuda')

class AGNewsDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=100):
        self.texts   = texts
        self.labels  = labels
        self.vocab   = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        tokens = word_tokenize(text.lower())[:self.max_len]
        indices = [self.vocab.get(token, self.vocab['<unk>']) for token in tokens]
        if len(indices) < self.max_len:
            indices += [self.vocab['<pad>']] * (self.max_len - len(indices))
        return torch.tensor(indices, dtype=torch.long), torch.tensor(label, dtype=torch.long)

In [6]:
from agnews_model import AGNewsDataset, build_vocab, load_glove_vectors, create_embedding_matrix, TextClassifier, predict_single

In [7]:
vocab = build_vocab(df_train['text'])
glove_vectors = load_glove_vectors('glove/glove.6B.100d.txt')
embedding_matrix = create_embedding_matrix(vocab, glove_vectors)

### Training

In [8]:
import pickle

def train_model(train_df, test_df, class_map, embed_matrix, vocab, model_save_path='config/agnews_model.pth',
                vocab_save_path='config/vocab.pkl', class_map_save_path='config/class_map.pkl', embed_dim=100,
                hidden_dim=128, batch_size=32, num_epochs=10, lr=0.001, max_len=100):
    model = TextClassifier(len(vocab), embed_dim, hidden_dim, len(class_map)).to(device)
    model.embedding.weight = nn.Parameter(embed_matrix.to(device), requires_grad=False)

    train_dataset = AGNewsDataset(train_df['text'].tolist(), train_df['Class Index'].tolist(), vocab, max_len)
    test_dataset = AGNewsDataset(test_df['text'].tolist(), test_df['Class Index'].tolist(), vocab, max_len)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for texts, labels in train_loader:
            texts, labels = texts.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(texts)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {total_loss / len(train_loader):.4f}")

    torch.save(model.state_dict(), model_save_path)
    with open(vocab_save_path, 'wb') as f:
        pickle.dump(vocab, f)
    with open(class_map_save_path, 'wb') as f:
        pickle.dump(class_map, f)
    print(f"模型已保存至 {model_save_path}")
    return model, test_loader

# 測試函數
def test_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for texts, labels in test_loader:
            texts, labels = texts.to(device), labels.to(device)
            outputs = model(texts)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")
    return accuracy

In [9]:
# 執行訓練與測試
model, test_loader = train_model(df_train, df_test, class_map, embedding_matrix, vocab)
test_model(model, test_loader)



Epoch 1, Loss: 1.0008
Epoch 2, Loss: 0.2952
Epoch 3, Loss: 0.2506
Epoch 4, Loss: 0.2110
Epoch 5, Loss: 0.1862
Epoch 6, Loss: 0.1676
Epoch 7, Loss: 0.1512
Epoch 8, Loss: 0.1346
Epoch 9, Loss: 0.1206
Epoch 10, Loss: 0.1079
模型已保存至 config/agnews_model.pth
Test Accuracy: 92.62%


92.61842105263158

In [10]:
# 單次預測
text = '''
The Race is On: Second Private Team Sets Launch Date for Human Spaceflight (SPACE.com),"SPACE.com - TORONTO, Canada -- A second\team of rocketeers competing for the  #36;10 million Ansari X Prize, a contest for\privately funded suborbital space flight, has officially announced the first\launch date for its manned rocket."
''' # Label = 3 , Sci/Tech
idx, label = predict_single(
            text=text,
            model_path='config/agnews_model.pth',
            vocab_path='config/vocab.pkl',
            class_map_path='config/class_map.pkl',
        )
print(f"Predicted Index: {idx}, Label: {label}")

Predicted Index: 3, Label: Sci/Tech


e:\Work_Space\WorkingSpace\AGNewsClassification\agnews_model.py:87: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=d